# LSTM vs Baseline Statistical Tests

Notebook ini menjalankan uji statistik pada prediksi test-set LSTM aktif dibandingkan baseline. Karena prediksi LSTM berasal dari checkpoint deep learning, workflow disimpan sebagai `.ipynb`.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = Path("C:/AIproject/AWAI")

sys.path.insert(0, str(PROJECT_ROOT / "src"))
PROJECT_ROOT

In [ ]:
from datetime import datetime
import json

import numpy as np
import pandas as pd
import torch

from traffic_prediction.config.settings import load_config
from traffic_prediction.data.processor import DataProcessor
from traffic_prediction.evaluation.baselines import (
    HistoricalAverageBaseline,
    LinearRegressionBaseline,
    PersistenceBaseline,
)
from traffic_prediction.evaluation.statistics import StatisticalTester
from traffic_prediction.features.offline import FeatureEngineer
from traffic_prediction.features.spatial import build_neighbor_mapping
from traffic_prediction.models.lstm import LSTMModelConfig, TrafficLSTM
from traffic_prediction.models.registry import ModelRegistry
from traffic_prediction.pipelines.offline import select_feature_columns

config = load_config(project_root=PROJECT_ROOT)
run_stamp = datetime.now().strftime("statistical-tests-%Y%m%d-%H%M%S")
report_dir = config.paths.reports_dir / run_stamp
report_dir.mkdir(parents=True, exist_ok=True)

registry = ModelRegistry(config.paths.models_dir / "registry.json")
active_model = registry.get_active()
if active_model is None or active_model.model_type != "lstm":
    raise RuntimeError("Active model must be a trained LSTM before running this notebook")

model_dir = Path(active_model.artifact_path)
model_config_payload = json.loads((model_dir / "model_config.json").read_text(encoding="utf-8"))
print("Statistical test run:", run_stamp)
print("Active model:", active_model.model_version)

In [ ]:
processor = DataProcessor(config.data, config.features)
traffic_raw, validation_report = processor.load_traffic_csv(config.paths.traffic_csv)
roads = processor.load_roads_csv(config.paths.roads_csv)
cleaned = processor.clean(traffic_raw)

neighbor_mapping = build_neighbor_mapping(
    roads,
    neighbor_count=config.features.spatial_neighbor_count,
)
feature_engineer = FeatureEngineer(
    neighbor_mapping=neighbor_mapping,
    lag_steps=config.features.lag_steps,
    rolling_windows=config.features.rolling_windows,
)
featured = feature_engineer.extract_features(cleaned)
feature_columns = select_feature_columns(featured)

train, validation, test, split_stats = processor.chronological_split(featured)
train_scaled = processor.fit_transform_train(train, feature_columns)
test_scaled = processor.transform_eval(test)

X_train, y_train, train_metadata = processor.create_sequences(train_scaled, feature_columns)
X_test, y_test, test_metadata = processor.create_sequences(test_scaled, feature_columns)

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_test:", X_test.shape, "y_test:", y_test.shape)

In [ ]:
model_config = LSTMModelConfig(**model_config_payload["model_config"])
checkpoint = torch.load(model_config_payload["checkpoint_path"], map_location="cpu")
model = TrafficLSTM(model_config)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

def predict_in_batches(X: np.ndarray, batch_size: int = 2048) -> np.ndarray:
    outputs = []
    with torch.no_grad():
        for start in range(0, len(X), batch_size):
            batch = torch.as_tensor(X[start : start + batch_size], dtype=torch.float32)
            outputs.append(model(batch).numpy())
    return np.concatenate(outputs, axis=0)

lstm_predictions_scaled = predict_in_batches(X_test)
lstm_predictions_kmh = processor.scaler_store.inverse_transform_speed(
    lstm_predictions_scaled.reshape(-1, 1)
).reshape(lstm_predictions_scaled.shape)
y_test_kmh = processor.scaler_store.inverse_transform_speed(y_test.reshape(-1, 1)).reshape(y_test.shape)

y_true_flat = y_test_kmh.reshape(-1)
lstm_pred_flat = lstm_predictions_kmh.reshape(-1)
len(y_true_flat)

In [ ]:
frequency = pd.Timedelta(config.data.frequency)
current_speed_index = feature_columns.index("current_speed")
baseline_feature_columns = [f"feature__{column}" for column in feature_columns]

def inverse_speed(values: np.ndarray) -> np.ndarray:
    return processor.scaler_store.inverse_transform_speed(values.reshape(-1, 1)).reshape(values.shape)

def build_horizon_frame(X: np.ndarray, y: np.ndarray, metadata) -> pd.DataFrame:
    y_kmh = inverse_speed(y)
    last_step_features = X[:, -1, :]
    lag_1_kmh = inverse_speed(X[:, -1, current_speed_index])
    rows = []
    for sequence_index, item in enumerate(metadata):
        for horizon_index in range(y.shape[1]):
            row = {
                "road_id": item.road_id,
                "timestamp": pd.Timestamp(item.target_start) + horizon_index * frequency,
                "horizon_minutes": int((horizon_index + 1) * frequency.total_seconds() / 60),
                "actual_speed": float(y_kmh[sequence_index, horizon_index, 0]),
                "lag_1": float(lag_1_kmh[sequence_index]),
            }
            for column, value in zip(baseline_feature_columns, last_step_features[sequence_index]):
                row[column] = float(value)
            rows.append(row)
    return pd.DataFrame(rows)

train_eval = build_horizon_frame(X_train, y_train, train_metadata)
test_eval = build_horizon_frame(X_test, y_test, test_metadata)

baselines = {
    "naive_persistence": PersistenceBaseline().fit(train_eval),
    "historical_average": HistoricalAverageBaseline().fit(train_eval),
    "linear_regression": LinearRegressionBaseline(feature_columns=baseline_feature_columns).fit(train_eval),
}
baseline_predictions = {
    name: baseline.predict(test_eval)
    for name, baseline in baselines.items()
}
{name: values.shape for name, values in baseline_predictions.items()}

In [ ]:
tester = StatisticalTester(alpha=0.05, confidence_level=0.95)
stats_table = tester.compare_against_many(
    y_true=y_true_flat,
    model_pred=lstm_pred_flat,
    baseline_predictions=baseline_predictions,
    horizon=config.features.horizon,
)

stats_table["winner_from_mean_difference"] = np.where(
    stats_table["confidence_interval"].apply(lambda item: item["mean"] > 0),
    "baseline",
    "lstm",
)
stats_table["mean_difference"] = stats_table["confidence_interval"].apply(lambda item: item["mean"])
stats_table["ci_lower"] = stats_table["confidence_interval"].apply(lambda item: item["lower"])
stats_table["ci_upper"] = stats_table["confidence_interval"].apply(lambda item: item["upper"])
stats_table = stats_table.drop(columns=["confidence_interval"])
stats_table

In [ ]:
summary = {
    "run_id": run_stamp,
    "active_model_version": active_model.model_version,
    "created_at": datetime.now().isoformat(),
    "alpha": 0.05,
    "confidence_level": 0.95,
    "sample_count": int(len(y_true_flat)),
    "horizon": int(config.features.horizon),
    "interpretation": "Positive mean_difference means LSTM has larger error/loss than the baseline.",
    "results": stats_table.to_dict(orient="records"),
}

csv_path = report_dir / "lstm_baseline_statistical_tests.csv"
json_path = report_dir / "lstm_baseline_statistical_tests.json"
stats_table.to_csv(csv_path, index=False)
json_path.write_text(json.dumps(summary, indent=2, default=str), encoding="utf-8")

print("CSV:", csv_path)
print("JSON:", json_path)
stats_table